# The PyBaMM model zoo

The model zoo is the home for battery models that live **outside PyBaMM's
core**: new physics from a paper, a partner's model, a teaching model. Each
model is a folder with its code, tests, examples, citation, and a short
`model.toml` manifest. Everything else (the registry, the test suite, the docs
page, the status badge) is generated from that manifest.

Why not just put new models in core?

* **Core stays small and maintainable.** Not every model needs to be maintained
  by the PyBaMM team forever.
* **Your model is still tested on every PyBaMM change**, so you find out when a
  release breaks it, and users can see a status badge.
* **You get credit.** Using a zoo model registers its citation automatically.

The zoo has two tiers: `community` models are tested on every pull request but
cannot block a PyBaMM release; `core` models are adopted by the maintainers and
gate merges.

In [1]:
import numpy as np

import pybamm
import pybamm_model_zoo as zoo

## 1. Browse the zoo

In [2]:
zoo.list_models()

['LinearisedSPM', 'MultiLayer3DThermalSPM']

In [3]:
entry = zoo.info("LinearisedSPM")
print(entry.title)
print(entry.summary)
print("tier:", entry.tier)
print("maintainers:", [m.github for m in entry.maintainers])
print("works with PyBaMM", entry.pybamm_requires)

Single Particle Model with a linearised open-circuit potential
SPM linearised about its starting stoichiometry, for fitting diffusivity to GITT pulses.
tier: core
maintainers: ['pybamm-team/maintainers']
works with PyBaMM >=26.0


## 2. Use a zoo model

`zoo.load` returns the model *class*, which you use like any PyBaMM model.
`LinearisedSPM` is an SPM whose open-circuit potentials are linearised about the
starting stoichiometry, which is exactly the assumption behind GITT analysis.

In [4]:
LinearisedSPM = zoo.load("LinearisedSPM")
model = LinearisedSPM()

linear = pybamm.Simulation(model).solve([0, 600])
full = pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 600])

t = np.linspace(0, 600, 7)
print("linearised SPM:", linear["Voltage [V]"](t))
print("SPM:           ", full["Voltage [V]"](t))

linearised SPM: [3.76005302 3.73995047 3.7271196  3.71472643 3.7023713  3.689986
 3.67754963]
SPM:            [3.78008056 3.76058016 3.7486904  3.73786398 3.72785951 3.7186925
 3.71036417]


The two agree at the start, and drift apart as the stoichiometry moves away from the point the model was linearised about.

In [5]:
pybamm.dynamic_plot(
    [linear, full],
    ["Voltage [V]", "X-averaged positive electrode open-circuit potential [V]"],
)

interactive(children=(FloatSlider(value=0.0, description='t', max=600.0, step=6.0), Output()), _dom_classes=('…

Using the model credits its authors:

In [6]:
pybamm.print_citations()

[1] Joel A. E. Andersson, Joris Gillis, Greg Horn, James B. Rawlings, and Moritz Diehl. CasADi – A software framework for nonlinear optimization and optimal control. Mathematical Programming Computation, 11(1):1–36, 2019. doi:10.1007/s12532-018-0139-4.
[2] Von DAG Bruggeman. Berechnung verschiedener physikalischer konstanten von heterogenen substanzen. i. dielektrizitätskonstanten und leitfähigkeiten der mischkörper aus isotropen substanzen. Annalen der physik, 416(7):636–664, 1935.
[3] Charles R. Harris, K. Jarrod Millman, Stéfan J. van der Walt, Ralf Gommers, Pauli Virtanen, David Cournapeau, Eric Wieser, Julian Taylor, Sebastian Berg, Nathaniel J. Smith, and others. Array programming with NumPy. Nature, 585(7825):357–362, 2020. doi:10.1038/s41586-020-2649-2.
[4] Alan C. Hindmarsh. The PVODE and IDA algorithms. Technical Report, Lawrence Livermore National Lab., CA (US), 2000. doi:10.2172/802599.
[5] Alan C. Hindmarsh, Peter N. Brown, Keith E. Grant, Steven L. Lee, Radu Serban, Dan E

## 3. Anatomy of a zoo model

Every zoo model is a folder with the same shape:

```
linearised_spm/
├── model.toml        # the manifest: the only boilerplate you write
├── __init__.py
├── model.py          # the physics
├── README.md         # Summary / Usage / Validation / Citation
├── CITATION.bib
├── tests/            # tests that pin a *physical* result
└── examples/         # scripts that are run in CI
```

In [7]:
for path in sorted(entry.path.rglob("*")):
    if "__pycache__" not in path.parts:
        print(path.relative_to(entry.path.parent))

linearised_spm/CITATION.bib
linearised_spm/README.md
linearised_spm/__init__.py
linearised_spm/examples
linearised_spm/examples/run_linearised_spm.py
linearised_spm/model.py
linearised_spm/model.toml
linearised_spm/tests
linearised_spm/tests/test_linearised_spm.py


The manifest says what the model is, who maintains it, which PyBaMM versions it supports, and how to test it:

In [8]:
print(entry.manifest_path.read_text())

[model]
slug = "linearised_spm"
name = "LinearisedSPM"
title = "Single Particle Model with a linearised open-circuit potential"
summary = "SPM linearised about its starting stoichiometry, for fitting diffusivity to GITT pulses."
class = "pybamm_model_zoo.linearised_spm:LinearisedSPM"
tier = "core"
pybamm_requires = ">=26.0"
added = "2026-08-20"
license = "BSD-3-Clause"

[[model.maintainers]]
name = "The PyBaMM Team"
github = "pybamm-team/maintainers"

[model.citation]
key = "PyBaMMModelZoo2026"

[model.tests]
# A GITT pulse, not a discharge: the linearisation is local to the starting
# stoichiometry, so a full discharge is outside what this model claims.
solve_time = 300
key_variables = [
    "Voltage [V]",
    "Positive electrode open-circuit potential gradient [V]",
]
skip_contract = []



## 4. The contract

Every model is checked against the same **contract**: the manifest is valid,
the folder has the right files, the model imports, registers its citation, is
well posed, builds, and solves with finite key variables. You only write
tests for your physics; the contract is free.

In [9]:
from pybamm_model_zoo.testing.contract import MODEL, checks_in_scope

for check in checks_in_scope(MODEL):
    check.run(entry)
    print(f"{check.name:12s} passed")

manifest     passed
layout       passed
import       passed
citation     passed
well_posed   passed
build        passed
solve        passed


## Exercises

1. Run `LinearisedSPM({"working electrode": "positive"})` for a 60 s pulse and
   plot the voltage against $\sqrt{t}$. Is it a straight line?
2. Read `examples/run_linearised_spm.py` in the folder printed above and run it.
3. Next notebook: write your own model in the zoo format.